In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant


In [24]:
def merge_stats(pg, ad):
    return pd.merge(pg, ad, on=["Season", "Team"], how="inner")

    
# Reload raw files without cleaning
PG_1_raw = pd.read_csv("data/2014-24pergamestats.csv")
AD_1_raw = pd.read_csv("data/2014-24advancedstats.csv")
PG_2_raw = pd.read_csv("data/2024-25pergamestats.csv")
AD_2_raw = pd.read_csv("data/2024-25advancedstats.csv")
PG_3_raw = pd.read_csv("data/2025-26pergamestats.csv")
AD_3_raw = pd.read_csv("data/2025-26advancedstats.csv")

PG_2_raw['Season'] = 2025
AD_2_raw['Season'] = 2025

PG_3_raw['Season'] = 2026
AD_3_raw['Season'] = 2026

# Merge raw without cleaning
mega_raw = pd.concat([
    merge_stats(PG_1_raw, AD_1_raw),
    merge_stats(PG_2_raw, AD_2_raw),
    merge_stats(PG_3_raw, AD_3_raw)
], ignore_index=True)

# Minimal prep only — no dropping, no filtering
mega_raw['Team'] = mega_raw['Team'].str.replace('*', '', regex=False).str.strip()
mega_raw = mega_raw.rename(columns={
    'eFG%':     'OFF_eFG%',
    'eFG%.1':   'DEF_eFG%',
    'TOV%':     'OFF_TOV%',
    'TOV%.1':   'DEF_TOV%',
    'FT/FGA':   'OFF_FT/FGA',
    'FT/FGA.1': 'DEF_FT/FGA'
})
mega_raw['pyth_diff'] = mega_raw['W'] - mega_raw['PW']

# Confirm leakage columns are present
leakage_check = ['MOV', 'SOS', 'SRS', 'ORtg', 'DRtg', 'NRtg', 'TS%']
print("Leakage columns present:", [c for c in leakage_check if c in mega_raw.columns])
print("Total columns:", mega_raw.shape[1])

Leakage columns present: ['MOV', 'SOS', 'SRS', 'ORtg', 'DRtg', 'NRtg', 'TS%']
Total columns: 52


In [27]:
features_original = [
    'G', 'Age', 'Pace', 'TS%', '3PAr',
    'ORB%', 'DRB%',
    'MOV', 'SOS', 'SRS',
    'ORtg', 'DRtg', 'NRtg',
    'OFF_eFG%', 'DEF_eFG%',
    'OFF_TOV%', 'DEF_TOV%',
    'OFF_FT/FGA', 'DEF_FT/FGA',
    'pyth_diff'
]

# Only use columns that exist
features_original = [f for f in features_original if f in mega_raw.columns]

vif_data_full = mega_raw[features_original].dropna()
X_full = add_constant(vif_data_full)

vif_df_full = pd.DataFrame({
    'Feature': features_original,
    'VIF': [variance_inflation_factor(X_full.values, i+1)
            for i in range(len(features_original))]
}).sort_values('VIF', ascending=False).reset_index(drop=True)

print(vif_df_full)



       Feature           VIF
0         DRtg           inf
1         NRtg           inf
2         ORtg           inf
3          MOV  1.073900e+06
4          SRS  1.016633e+06
5          SOS  6.094113e+03
6          TS%  9.791913e+02
7     OFF_eFG%  1.934048e+02
8     DEF_eFG%  1.640235e+02
9         ORB%  9.016445e+01
10    OFF_TOV%  7.211310e+01
11    DEF_TOV%  3.652144e+01
12        DRB%  1.904240e+01
13        3PAr  7.830298e+00
14  DEF_FT/FGA  7.442254e+00
15  OFF_FT/FGA  6.722531e+00
16        Pace  2.431041e+00
17         Age  1.621859e+00
18           G  1.346510e+00
19   pyth_diff  1.191199e+00


/Users/travisowusu/miniforge3/envs/project/lib/python3.14/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


In [6]:
df1 = pd.read_csv("data/2014-24pergamestats.csv")
df2 = pd.read_csv("data/2014-24advancedstats.csv")
df3 = pd.read_csv("data/2024-25pergamestats.csv")
df4 = pd.read_csv("data/2024-25advancedstats.csv")
df5 = pd.read_csv("data/2025-26pergamestats.csv")
df6 = pd.read_csv("data/2025-26advancedstats.csv")

df2.columns

Index(['Season', 'Team', 'Age', 'W', 'L', 'PW', 'PL', 'MOV', 'SOS', 'SRS',
       'ORtg', 'DRtg', 'NRtg', 'Pace', 'FTr', '3PAr', 'TS%', 'eFG%', 'TOV%',
       'ORB%', 'FT/FGA', 'eFG%.1', 'TOV%.1', 'DRB%', 'FT/FGA.1'],
      dtype='str')

In [7]:

def clean_stats(df, s):
    
    # 1. Add season if missing
    if 'Season' not in df.columns:
        df['Season'] = s
    df['Season'] = df['Season'].astype(int)

    # 2. Clean team names
    df['Team'] = df['Team'].str.replace('*', '', regex=False).str.strip()

    # 3. Rename BEFORE filtering columns
    rename_map = {
        'eFG%':     'OFF_eFG%',
        'eFG%.1':   'DEF_eFG%',
        'TOV%':     'OFF_TOV%',
        'TOV%.1':   'DEF_TOV%',
        'FT/FGA':   'OFF_FT/FGA',
        'FT/FGA.1': 'DEF_FT/FGA'
    }
    df = df.rename(columns=rename_map)

    # 4. Keep only what we need — no leakage, no redundant raw counts
    keep_columns = [
        'Season', 'Team', 'G', 'Age',
        'W', 'L', 'PW', 'PL',
        'Pace', 'FTr', '3PAr',
        'OFF_eFG%', 'DEF_eFG%',
        'OFF_TOV%', 'DEF_TOV%',
        'ORB%',     'DRB%',
        'OFF_FT/FGA', 'DEF_FT/FGA'
    ]
    df = df[[col for col in keep_columns if col in df.columns]]

    # 5. Enforce numeric types
    numeric_cols = [c for c in df.columns if c != 'Team']
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

    # 6. Engineer pyth_diff
    if 'PW' in df.columns and 'W' in df.columns:
        df['pyth_diff'] = df['W'] - df['PW']

    # 7. Drop duplicates
    df = df.drop_duplicates(subset=['Team', 'Season'], keep='first')

    # 8. Drop rows missing any core Four Factors
    four_factors = [
        'OFF_eFG%', 'DEF_eFG%',
        'OFF_TOV%', 'DEF_TOV%',
        'ORB%',     'DRB%',
        'OFF_FT/FGA', 'DEF_FT/FGA'
    ]
    existing_ff = [c for c in four_factors if c in df.columns]
    df = df.dropna(subset=existing_ff, how='any')

    return df




In [8]:
PG_1 = clean_stats(df1, s=None)
AD_1 = clean_stats(df2, s=None)
PG_2 = clean_stats(df3, 2025)
AD_2 = clean_stats(df4, 2025)
PG_3 = clean_stats(df5, 2026)
AD_3 = clean_stats(df6, 2026)

In [9]:
PG_3

,Season,Team,G
0,2026,Denver Nuggets,82
1,2026,Miami Heat,82
2,2026,San Antonio Spurs,82
3,2026,Cleveland Cavaliers,82
4,2026,Oklahoma City Thunder,82
5,2026,Atlanta Hawks,82
6,2026,Minnesota Timberwolves,82
7,2026,Detroit Pistons,82
8,2026,Utah Jazz,82
9,2026,New York Knicks,82


In [10]:
o = AD_3['Team'].unique()
len(o)

30

In [11]:
p = PG_3['Team'].unique()
len(p)

30

In [12]:
both = []
for i in o:
    if i not in p:
       print(i)


        

In [13]:
s_12_24 = merge_stats(PG_1, AD_1)
s_25    = merge_stats(PG_2, AD_2)
s_26    = merge_stats(PG_3, AD_3)

In [14]:
mega = pd.concat([s_12_24, s_25, s_26], ignore_index=True)

mega.columns

Index(['Season', 'Team', 'G', 'Age', 'W', 'L', 'PW', 'PL', 'Pace', 'FTr',
       '3PAr', 'OFF_eFG%', 'DEF_eFG%', 'OFF_TOV%', 'DEF_TOV%', 'ORB%', 'DRB%',
       'OFF_FT/FGA', 'DEF_FT/FGA', 'pyth_diff'],
      dtype='str')

In [15]:
mega.to_csv('data/mega.csv', index=False)

print(f"Saved: {mega.shape[0]} rows x {mega.shape[1]} columns")
print(f"Seasons: {mega['Season'].min()} - {mega['Season'].max()}")
print(f"Teams per season:\n{mega.groupby('Season').size()}")

Saved: 390 rows x 20 columns
Seasons: 2014 - 2026
Teams per season:
Season
2014    30
2015    30
2016    30
2017    30
2018    30
2019    30
2020    30
2021    30
2022    30
2023    30
2024    30
2025    30
2026    30
dtype: int64


In [16]:
# print(mega[['FTr', 'OFF_FT/FGA']].corr())
# # If correlation = 1.0, drop one of them
# mega = mega.drop(columns=['FTr'])


In [17]:
mega.columns

Index(['Season', 'Team', 'G', 'Age', 'W', 'L', 'PW', 'PL', 'Pace', 'FTr',
       '3PAr', 'OFF_eFG%', 'DEF_eFG%', 'OFF_TOV%', 'DEF_TOV%', 'ORB%', 'DRB%',
       'OFF_FT/FGA', 'DEF_FT/FGA', 'pyth_diff'],
      dtype='str')